In [1]:
import torch
import torch.nn as nn

In [ ]:
class CounterDetectionNetwork(nn.Module):
    def __init__(self, num_classes=1000):
        super(CounterDetectionNetwork, self).__init__()
        self.conv1 = conv_block(3, 64, kernel_size=3, stride=1, padding=1)
        self.max1 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.conv2 = conv_block(64, 64, kernel_size=1, stride=1, padding=1)
        self.conv3 = conv_block(64, 64, kernel_size=3, stride=1, padding=1)
        self.conv4 = conv_block(64, 64, kernel_size=3, stride=2, padding=1)
        self.max2 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.incept1 = Inception_block(256, 64, 96, 128, 16, 32, 32)
        self.max3 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.incept2 = Inception_block(480, 128, 128, 192, 32, 96, 64)
        self.conv5 = conv_block(48, 128, kernel_size=1, stride=1, padding=1)
        self.conv6 = conv_block(128, 128, kernel_size=3, stride=1, padding=1)
        self.conv7 = conv_block(128, 128, kernel_size=3, stride=2, padding=1)
        self.incept3 = Inception_block(1024, 352, 192, 320, 160, 224, 128)
        self.incept4 = Inception_block(1024, 352, 192, 320, 160, 224, 128)
        self.conv8 = conv_block(96, 96, kernel_size=1, stride=1, padding=1)
        self.conv9 = conv_block(96, 96, kernel_size=3, stride=1, padding=1)
        self.conv10 = conv_block(96, 512, kernel_size=3, stride=1, padding=1)
        


class Inception_block(nn.Module):
    def __init__(self, in_channels, out_channels_1x1, red_channels_3x3, out_channels_3x3, red_channels_5x5, out_channels_5x5, pool_proj):
        super(Inception_block, self).__init__()

        self.branch1 = conv_block(in_channels, out_channels_1x1, kernel_size=1)

        self.branch2 = nn.Sequential(
            conv_block(in_channels, red_channels_3x3, kernel_size=1),
            conv_block(red_channels_3x3, out_channels_3x3, kernel_size=3, padding=1)
        )
        self.branch3 = nn.Sequential(
            conv_block(in_channels, red_channels_5x5, kernel_size=1),
            conv_block(red_channels_5x5, out_channels_5x5, kernel_size=5, padding=2)
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            conv_block(in_channels,pool_proj, kernel_size=1)
        )

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], 1)



class conv_block(nn.Module):
    def __init__(self, in_channels, out_channels, **kwargs):
        super(conv_block, self).__init__()
        
        self.conv = nn.Conv2d(in_channels, out_channels, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))